# Classify Intents with Encoders

Requests to a voice assistant ("wake me up at nine", "play some jazz"), each with one of 60 intents: a real classification problem, with many classes, imbalanced and sometimes close. The data: Amazon's [MASSIVE](https://github.com/alexa/massive).

1. Classic models on TF-IDF: the baseline
2. Frozen *embeddings* + logistic regression
3. Fine-tune an encoder
4. Keep the results for tomorrow (the next lab compares with an LLM)
5. Bonus: SetFit with 8 examples per class, and calibration

This notebook needs a GPU: *Runtime > Change runtime type > T4 GPU*.

## Data

In [ ]:
!wget -qO- https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh | bash -s massive-intents

In [ ]:
import time
from collections import Counter

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score

train = pd.read_parquet("massive-intents/en-train.parquet")
test = pd.read_parquet("massive-intents/en-test.parquet")
labels = sorted(train.label.unique())
print(len(train), "training examples,", len(test), "test,", len(labels), "intents")
train.label.value_counts().describe()

`record` measures and keeps each method's results: the macro F1 (every intent counts the same), the accuracy, the training time and the prediction time per text. `confusions` shows the pairs of intents most often confused.

In [ ]:
results = {}


def record(name: str, predictions, train_seconds: float, predict_seconds: float) -> None:
    results[name] = {
        "predictions": list(predictions),
        "train_s": train_seconds,
        "ms_per_text": 1000 * predict_seconds / len(test),
    }
    print(
        f"{name:30} macro F1 {f1_score(test.label, predictions, average='macro'):.3f}"
        f"  accuracy {accuracy_score(test.label, predictions):.3f}"
        f"  training {train_seconds:5.0f} s  {results[name]['ms_per_text']:.2f} ms/text"
    )


def confusions(predictions, n: int = 10) -> list[tuple[tuple[str, str], int]]:
    errors = Counter((t, p) for t, p in zip(test.label, predictions) if t != p)
    return errors.most_common(n)

## 1. The Baseline: Classic Models on TF-IDF

The official program's models, on a TF-IDF representation (words and word pairs). Run the cell and read the results: which one is best? Which intents does it confuse?

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC

vectorizer = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, min_df=2)
X_train = vectorizer.fit_transform(train.text)
X_test = vectorizer.transform(test.text)
for name, model in (
    ("TF-IDF + Naive Bayes", MultinomialNB(alpha=0.1)),
    ("TF-IDF + linear SVM", LinearSVC()),
    ("TF-IDF + logistic regression", LogisticRegression(max_iter=2000, C=10)),
    ("TF-IDF + random forest", RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=0)),
):
    start = time.time()
    model.fit(X_train, train.label)
    trained = time.time() - start
    start = time.time()
    predictions = model.predict(X_test)
    record(name, predictions, trained, time.time() - start)

# Trees (forests, gradient boosting), the kings of tabular data, do poorly on thousands of
# sparse columns: gradient boosting (LightGBM) does even worse, and diverges here. On text,
# the classic baseline is a linear model on TF-IDF.

In [ ]:
confusions(results["TF-IDF + linear SVM"]["predictions"])

## 2. Frozen *Embeddings* + Logistic Regression

Encode the training and test texts with Qwen3-Embedding-0.6B (`model.encode(texts, prompt=INSTRUCTION, batch_size=64)`), each text preceded by the instruction below, which describes the task to the model. Train a logistic regression on the vectors, and record the result with `record` (the prediction time includes encoding the test texts).

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B", device="cuda").half()
embedder.max_seq_length = 64
INSTRUCTION = "Instruct: Classify the intent of a request to a voice assistant\nQuery: "

In [ ]:
# Your code here

### Solution

In [ ]:
start = time.time()
E_train = embedder.encode(train.text.tolist(), prompt=INSTRUCTION, batch_size=64)
classifier = LogisticRegression(max_iter=2000, C=10).fit(E_train, train.label)
trained = time.time() - start
start = time.time()
E_test = embedder.encode(test.text.tolist(), prompt=INSTRUCTION, batch_size=64)
predictions = classifier.predict(E_test)
record("Embeddings + regression", predictions, trained, time.time() - start)

## 3. Fine-Tune an Encoder

Fine-tune `ENCODER` (a pretrained encoder, plus a classification layer) on the training data, with `transformers`:

- tokenize the texts (`max_length=64`) and number the intents
- `AutoModelForSequenceClassification.from_pretrained(ENCODER, num_labels=60)`
- `TrainingArguments(num_train_epochs=3, learning_rate=5e-5, per_device_train_batch_size=32, fp16=True, warmup_steps=0.1, save_strategy="no", report_to="none")`
- a `Trainer` with a `DataCollatorWithPadding`

Training takes a few minutes: start it, and meanwhile read the results of steps 1 and 2 again. Then predict the test intents (`trainer.predict`) and record the result.

In [ ]:
ENCODER = "jhu-clsp/mmBERT-base"  # multilingual, 2025; on par with ModernBERT on this data

In [ ]:
# Your code here

### Solution

In [ ]:
from datasets import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

index = {label: i for i, label in enumerate(labels)}
tokenizer = AutoTokenizer.from_pretrained(ENCODER)


def to_dataset(df: pd.DataFrame) -> Dataset:
    ds = Dataset.from_dict({"text": df.text.tolist(), "label": [index[label] for label in df.label]})
    return ds.map(lambda batch: tokenizer(batch["text"], truncation=True, max_length=64), batched=True, remove_columns=["text"])


model = AutoModelForSequenceClassification.from_pretrained(ENCODER, num_labels=len(labels))
args = TrainingArguments(
    "encoder",
    num_train_epochs=3,
    learning_rate=5e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=128,
    fp16=True,
    warmup_steps=0.1,
    logging_steps=100,
    save_strategy="no",
    report_to="none",
)
trainer = Trainer(model=model, args=args, train_dataset=to_dataset(train), data_collator=DataCollatorWithPadding(tokenizer))
start = time.time()
trainer.train()
trained = time.time() - start

start = time.time()
logits = trainer.predict(to_dataset(test)).predictions
predictions = [labels[i] for i in logits.argmax(axis=1)]
record("Fine-tuned encoder", predictions, trained, time.time() - start)

## 4. Keep the Results for Tomorrow

Tomorrow, the next lab compares these methods with an LLM. Save each method's predictions to `predictions-en.csv` (one row per test text: `id`, `label`, then one column per method) and the timings to `timings-en.csv`, then download both files (*Files* panel on the left, or `google.colab.files.download`). A copy of our results is published, just in case.

In [ ]:
# Your code here

### Solution

In [ ]:
predictions = pd.DataFrame({"id": test.id, "label": test.label})
for name, result in results.items():
    predictions[name] = result["predictions"]
predictions.to_csv("predictions-en.csv", index=False)
timings = pd.DataFrame(
    [(name, r["train_s"], r["ms_per_text"]) for name, r in results.items()],
    columns=["method", "train_s", "ms_per_text"],
)
timings.to_csv("timings-en.csv", index=False)
timings

## 5. Bonus

**SetFit.** With only 8 examples per intent, fine-tune an *embedding* model contrastively, then a classification head ([SetFit](https://github.com/huggingface/setfit)). Where does it land, with 20 times fewer labels?

In [ ]:
# Your code here

### Solution

In [ ]:
%pip install -q setfit
from datasets import Dataset as HFDataset
from setfit import SetFitModel, Trainer as SetFitTrainer, TrainingArguments as SetFitArguments

few = train.groupby("label").sample(8, random_state=0, replace=True).drop_duplicates()
setfit_model = SetFitModel.from_pretrained("intfloat/multilingual-e5-base", labels=labels)
setfit_trainer = SetFitTrainer(
    model=setfit_model,
    args=SetFitArguments(batch_size=32, num_epochs=1, num_iterations=10, report_to="none"),
    train_dataset=HFDataset.from_dict({"text": ("query: " + few.text).tolist(), "label": few.label.tolist()}),
)
start = time.time()
setfit_trainer.train()
trained = time.time() - start
start = time.time()
predictions = setfit_model.predict(("query: " + test.text).tolist())
record("SetFit (8 per class)", list(predictions), trained, time.time() - start)

**Calibration.** For the regression on *embeddings* and the fine-tuned encoder, plot the reliability diagram: the share of correct predictions as a function of the announced probability (`sklearn.calibration.calibration_curve`, on "is the prediction correct?" and the predicted class's probability). Which one is better calibrated?

In [ ]:
# Your code here

### Solution

In [ ]:
import matplotlib.pyplot as plt
from scipy.special import softmax
from sklearn.calibration import calibration_curve

probabilities = {
    "Embeddings + regression": classifier.predict_proba(E_test),
    "Fine-tuned encoder": softmax(logits, axis=1),
}
for name, proba in probabilities.items():
    classes = classifier.classes_ if name.startswith("Embeddings") else np.array(labels)
    confidence = proba.max(axis=1)
    correct = classes[proba.argmax(axis=1)] == test.label.to_numpy()
    accuracy, mean_confidence = calibration_curve(correct, confidence, n_bins=10, strategy="quantile")
    plt.plot(mean_confidence, accuracy, "o-", label=name)
plt.plot([0, 1], [0, 1], "--", color="gray", label="Perfect calibration")
plt.xlabel("Announced probability")
plt.ylabel("Share of correct predictions")
plt.legend()
plt.show()